# SR-VAE (paper architecture), whole-image version for BioSR: LR 128x128 -> SR 256x256, conv widths / 8

**No patches, no tiling, no stitching.** Every training sample is a full widefield image (128 x 128) and its SIM ground truth (256 x 256).
The layout is the base paper's Fig. 2 / 4(b) / 8, adapted to the dataset: 2x (HR branch with one stride-2 block, one UBP block), 1 grayscale channel,
and **one deliberate change: the encoder conv channel widths are divided by `ch_div = 8`** (32, 64, 128, 256 become 4, 8, 16, 32).

| | Paper | This notebook |
|---|---|---|
| Scale / channels | 8x, RGB | 2x, grayscale |
| Model input | patches | **whole** 128 x 128 LR image and **whole** 256 x 256 HR image |
| Encoder / LR-stack conv widths | 32, 64, 128, 256 | **4, 8, 16, 32** (`ch_div = 8`; set 1 for the paper's widths) |
| Encoder FC | flatten -> FC 4096 -> FC 2048 | same; flatten size = 32 x 128 x 128 = 524,288 (paper widths: 4,194,304) |
| Generator | LR flattened into FC 1024 | whole LR image (16,384 values) flattened into FC 1024, no pooling |
| Latent map in decoder | 16 x 16 | 16 x 16 resized to 128 x 128 |

The decoder width `nf` stays 256. **Size (my arithmetic):** about 2.2 B parameters, 8.8 GB of weights in fp32, about 35 GB with gradients and Adam,
about 26 GB per full checkpoint. The first encoder FC layer alone has 2.15 B parameters. The sanity cell (section 8) reports the exact numbers without
allocating memory, and stops with a clear message if the weights cannot fit in GPU memory.

## 0. Install (run once)

In [ ]:
# %pip install torch torchvision numpy scikit-image mrcfile tifffile matplotlib

## 1. Imports

In [ ]:
import glob, math, os, random, time
from types import SimpleNamespace

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt

## 2. Data

Expected layout: `<data_root>/<structure>/{training_wf, training_gt, validate_wf, validate_gt}/`. Images are read and percentile-normalised to [0, 1] on demand, inside the DataLoader.

In [ ]:
def read_stack(path):
    """Read .mrc / .tif / .npy into a float32 numpy array."""
    ext = os.path.splitext(path)[1].lower()
    if ext in (".mrc", ".rec", ".st"):
        import mrcfile
        with mrcfile.open(path, permissive=True) as m:
            return np.array(m.data, dtype=np.float32)
    if ext in (".tif", ".tiff"):
        import tifffile
        return tifffile.imread(path).astype(np.float32)
    if ext == ".npy":
        return np.load(path).astype(np.float32)
    if ext in (".png", ".jpg", ".jpeg", ".bmp"):
        from skimage import io
        a = io.imread(path).astype(np.float32)
        return a[..., :3].mean(axis=-1) if a.ndim == 3 else a      # RGB -> gray
    raise ValueError(f"Unsupported file type: {path}")


def percentile_norm(img, lo=0.03, hi=99.7):
    """Robust intensity normalisation to [0,1]:  x' = clip((x - p_lo) / (p_hi - p_lo), 0, 1)."""
    a, b = np.percentile(img, [lo, hi])
    return np.clip((img - a) / (b - a + 1e-8), 0.0, 1.0).astype(np.float32)


IMG_EXTS = (".tif", ".tiff", ".png", ".mrc", ".npy")


def list_images(folder):
    return sorted(f for f in glob.glob(os.path.join(folder, "*")) if f.lower().endswith(IMG_EXTS))

In [ ]:
class FullImageDataset(Dataset):
    """Whole image pairs, no cropping: LR 128x128 and GT 256x256. Augmentation: random rot90 and flip (same for both)."""

    def __init__(self, pairs, augment=True):
        self.pairs, self.augment = pairs, augment

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, i):
        lr, hr = self.pairs[i]                      # read + percentile-normalised on demand
        if self.augment:
            if lr.shape[0] == lr.shape[1]:          # rot90 only keeps shapes for square images
                k = random.randint(0, 3)
                lr, hr = np.rot90(lr, k), np.rot90(hr, k)
            if random.random() < 0.5:
                lr, hr = lr[:, ::-1], hr[:, ::-1]
        return (torch.from_numpy(np.ascontiguousarray(lr))[None],
                torch.from_numpy(np.ascontiguousarray(hr))[None])

In [ ]:
class LazyPairs:
    """
    List-like object of (widefield, SIM) pairs. Stores file paths only.
        pairs[i]      -> reads both files, averages frames if needed, percentile-normalises, returns (lr, hr)
        pairs[a:b]    -> list of loaded pairs (used by evaluate(), which does pairs[:max_images])
        len(pairs)    -> number of pairs
    Indexing (done by the DataLoader dataset) is what triggers the disk read.
    """

    def __init__(self, wf_dir, gt_dir):
        wf, gt = list_images(wf_dir), list_images(gt_dir)
        if not wf or not gt:
            raise RuntimeError(f"No images found in {wf_dir} or {gt_dir}")
        stem = lambda p: os.path.splitext(os.path.basename(p))[0]
        gt_by_stem = {stem(p): p for p in gt}
        self.files = [(p, gt_by_stem[stem(p)]) for p in wf if stem(p) in gt_by_stem]
        if not self.files:
            print(f"[warn] no matching file names in {wf_dir} / {gt_dir}; pairing by sorted order")
            self.files = list(zip(wf, gt))

    def __len__(self):
        return len(self.files)

    def _load(self, i):
        lp, hp = self.files[i]
        lr = read_stack(lp)
        while lr.ndim > 2:                 # stack of raw frames -> widefield average
            lr = lr.mean(axis=0)
        hr = read_stack(hp)
        while hr.ndim > 2:                 # first frame of the SIM stack
            hr = hr[0]
        return percentile_norm(lr), percentile_norm(hr)

    def __getitem__(self, i):
        if isinstance(i, slice):
            return [self._load(j) for j in range(*i.indices(len(self)))]
        return self._load(i)

    def __iter__(self):
        return (self._load(j) for j in range(len(self)))

## 3. Model

### 3.0 Enhanced UBP block (Fig. 4b), 2x per block

In [ ]:
class EnhancedUBP(nn.Module):
    """
    Enhanced Up-sampling Back-Projection block (Fig. 4b), 2x per block.
    Back-projection idea (DBPN): up-project, down-project back, and use the LR-space
    error to correct the HR estimate.
        h0 = PReLU( Deconv(x) )                 # HR estimate            (H -> 2H)
        l0 = Conv(h0)                           # project back to LR     (2H -> H)
        r  = l0 - x                             # LR-space residual
        h1 = Deconv( Conv1x1_a(r) )             # residual mapped to HR (1x1 conv = weighted shortcut)
        y  = h0 + Conv1x1_b(h1)                 # weighted addition = residual update
    Paper: 6x6 kernels, stride 2 (pad 2 -> exact 2x), 1x1 convs for weighting, and one
    activation after the first deconv only (the second one was removed as redundant).
    (The exact wiring of the two 1x1 convs is our reading of Fig. 4b.)
    """

    def __init__(self, nf):
        super().__init__()
        self.up1 = nn.ConvTranspose2d(nf, nf, 6, 2, 2)
        self.act = nn.PReLU(nf)
        self.down = nn.Conv2d(nf, nf, 6, 2, 2)
        self.res_1x1 = nn.Conv2d(nf, nf, 1)
        self.up2 = nn.ConvTranspose2d(nf, nf, 6, 2, 2)
        self.out_1x1 = nn.Conv2d(nf, nf, 1)

    def forward(self, x):
        h0 = self.act(self.up1(x))
        r = self.down(h0) - x
        h1 = self.up2(self.res_1x1(r))
        return h0 + self.out_1x1(h1)

### 3.1 HR branch of the LR-HR encoder   (Fig. 2, left conv column)   [unchanged]

In [ ]:
ACT_BETWEEN_BLOCKS = False   # Fig. 2 draws a ReLU only BETWEEN the two convs of each block (False = exactly the figure).


class HRBranch(nn.Module):
    """
    Encoder branch that takes the HR image Y.  Three blocks, each:  Conv3x3 (stride 1) -> ReLU -> Conv3x3 (stride 1 or 2)
        block 1: in -> 32  -> 64      block 2: 64 -> 64  -> 128      block 3: 128 -> 128 -> 256     (all widths divided by ch_div)
    Paper (8x SR): the second conv of EVERY block has stride 2 -> Y is down-sampled 8x and lands on the LR size.
    Here (2x SR):  only the first block has stride 2 -> Y (256x256) is down-sampled 2x to the LR size (128x128).
    """

    def __init__(self, in_ch, scale, ch_div=1):
        super().__init__()
        n_down = int(math.log2(scale))
        w = lambda c: max(1, c // ch_div)                  # channel width = paper width / ch_div  (paper: ch_div = 1)
        layers, c_in = [], in_ch
        for b, (c1, c2) in enumerate([(32, 64), (64, 128), (128, 256)]):
            c1, c2 = w(c1), w(c2)
            s2 = 2 if b < n_down else 1
            layers += [nn.Conv2d(c_in, c1, 3, 1, 1), nn.ReLU(inplace=True), nn.Conv2d(c1, c2, 3, s2, 1)]
            if ACT_BETWEEN_BLOCKS and b < 2:
                layers.append(nn.ReLU(inplace=True))
            c_in = c2
        self.net = nn.Sequential(*layers)

    def forward(self, y):
        return self.net(y)

### 3.2 LR stack, SHARED by the encoder and the decoder   [unchanged]

In [ ]:
class LRStack(nn.Module):
    """
    Conv stack on the ORIGINAL LR image X, all stride 1:  block 1: in -> 32 -> 64 ;  block 2: 64 -> 128 -> 256  (widths / ch_div).
    Output (256 / ch_div channels, LR size) goes to the encoder (added to the HR branch) and to the decoder ("Addition" layer).
    """

    def __init__(self, in_ch, ch_div=1):
        super().__init__()
        w = lambda c: max(1, c // ch_div)
        layers, c_in = [], in_ch
        for b, (c1, c2) in enumerate([(32, 64), (128, 256)]):
            c1, c2 = w(c1), w(c2)
            layers += [nn.Conv2d(c_in, c1, 3, 1, 1), nn.ReLU(inplace=True), nn.Conv2d(c1, c2, 3, 1, 1)]
            if ACT_BETWEEN_BLOCKS and b < 1:
                layers.append(nn.ReLU(inplace=True))
            c_in = c2
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

### 3.3 LR-HR encoder head   (Fig. 2: circled +, FC 4096, ReLU, FC 2048, mu 1024, Sigma 1024)

In [ ]:
class LRHREncoder(nn.Module):
    """
    f = FC2048( ReLU( FC4096( flatten( HRBranch(Y) + LRStack(X) ) ) ) )      (+ = element-wise sum, C x L x L, C = 256 / ch_div)
    mu = Linear(2048, z_dim),  logvar = Linear(2048, z_dim)                   (paper: Sigma with Exp; sigma = exp(0.5 * logvar))
    Both branches work at LR resolution (L = 128), so the flattened size is C * L * L. With ch_div = 1 (paper widths) C = 256 and the
    first FC layer is Linear(4,194,304 -> 4096), about 17.2 B parameters. With ch_div = 8, C = 32 and it is Linear(524,288 -> 4096), about 2.15 B.
    """

    def __init__(self, lr_size, in_ch, scale, z_dim, ch_div=1):
        super().__init__()
        self.hr_branch = HRBranch(in_ch, scale, ch_div)
        c_last = max(1, 256 // ch_div)                     # channels after the branches (paper: 256)
        flat = c_last * lr_size * lr_size
        self.fc = nn.Sequential(nn.Flatten(), nn.Linear(flat, 4096), nn.ReLU(inplace=True), nn.Linear(4096, 2048))
        self.mu = nn.Linear(2048, z_dim)
        self.logvar = nn.Linear(2048, z_dim)

    def forward(self, hr, lr_feat):
        f = self.fc(self.hr_branch(hr) + lr_feat)
        return self.mu(f), self.logvar(f).clamp(-10.0, 10.0)

### 3.4 Conditional sampling generator G(X)   [only the input size changed: whole 128 x 128 image, no pooling]

In [ ]:
class ConditionalSamplingGenerator(nn.Module):
    """
    g = Normalize( FC_z( ReLU( FC_1024( Normalize( flatten(X) ) ) ) ) )
    The whole LR image (in_ch x L x L = 16,384 values for 128 x 128) is flattened into FC 1024. No pooling.
    Normalize(v) = (v - mean(v)) / std(v) per sample.
    """

    def __init__(self, lr_size, in_ch, z_dim):
        super().__init__()
        self.size = lr_size
        self.fc1 = nn.Linear(in_ch * lr_size * lr_size, 1024)
        self.fc2 = nn.Linear(1024, z_dim)

    @staticmethod
    def _norm(v, eps=1e-6):
        return (v - v.mean(1, keepdim=True)) / (v.std(1, keepdim=True) + eps)

    def forward(self, lr):
        assert lr.shape[-2:] == (self.size, self.size), \
            f"generator expects {self.size}x{self.size} LR input, got {tuple(lr.shape[-2:])}"
        x = self._norm(lr.flatten(1))
        return self._norm(self.fc2(F.relu(self.fc1(x))))

### 3.5 SR decoder   [unchanged]

In [ ]:
class SRDecoder(nn.Module):
    """
        latent path : z -> FC 2048 -> PReLU -> FC 4096 -> reshape 16x16x16 -> resize to the LR size -> Conv 3x3, nf
        LR path     : LRStack feature (256 ch) -> ReLU -> Conv 3x3, nf
        Addition    : latent path + LR path
        UBP blocks  : log2(scale) enhanced UBP blocks (2x: one block)
        output      : Conv 3x3 -> in_ch channels, then + Bicubic(X)
    For a 128x128 LR image the 16x16 latent map is resized (bilinear) to 128x128.
    """

    def __init__(self, z_dim, nf, scale, in_ch, ch_div=1, lat_ch=16, lat_hw=16):
        super().__init__()
        assert scale in (2, 4, 8), "scale must be 2, 4 or 8 (the HR branch has 3 blocks)"
        self.scale, self.lat_ch, self.lat_hw = scale, lat_ch, lat_hw
        self.fc = nn.Sequential(nn.Linear(z_dim, 2048), nn.PReLU(), nn.Linear(2048, lat_ch * lat_hw * lat_hw))
        self.lat_conv = nn.Conv2d(lat_ch, nf, 3, 1, 1)
        self.lr_conv = nn.Conv2d(max(1, 256 // ch_div), nf, 3, 1, 1)      # LR-stack features -> nf channels
        self.ubps = nn.Sequential(*[EnhancedUBP(nf) for _ in range(int(math.log2(scale)))])
        self.tail = nn.Conv2d(nf, in_ch, 3, 1, 1)

    def forward(self, z, lr, lr_feat):
        lat = self.fc(z).view(z.size(0), self.lat_ch, self.lat_hw, self.lat_hw)
        lat = F.interpolate(lat, size=lr_feat.shape[-2:], mode="bilinear", align_corners=False)
        x = self.lr_conv(F.relu(lr_feat)) + self.lat_conv(lat)                       # "Addition"
        x = self.tail(self.ubps(x))
        return x + F.interpolate(lr, scale_factor=self.scale, mode="bicubic", align_corners=False)

### 3.6 Full SR-VAE

In [ ]:
class SRVAE(nn.Module):
    """
    Training (needs Y):   lr_feat = LRStack(X);  (mu, logvar) = Enc(Y, lr_feat);  g = G(X)
                          z = mu + sigma * g ,  sigma = exp(0.5 * logvar)
                          Y' = Dec(z, X, lr_feat)
    Testing (no Y):       z = g = G(X)  ->  Y' = Dec(z, X, lr_feat)
    Works on whole images: X is lr_size x lr_size (128 x 128), Y is scale * lr_size (256 x 256).
    """

    def __init__(self, scale=2, lr_size=128, in_ch=1, z_dim=1024, nf=256, ch_div=1):
        super().__init__()
        self.scale, self.lr_size, self.ch_div = scale, lr_size, ch_div
        self.lr_stack = LRStack(in_ch, ch_div)
        self.enc = LRHREncoder(lr_size, in_ch, scale, z_dim, ch_div)
        self.gen = ConditionalSamplingGenerator(lr_size, in_ch, z_dim)
        self.dec = SRDecoder(z_dim, nf, scale, in_ch, ch_div)

    def forward(self, lr, hr=None):
        lr_feat = self.lr_stack(lr)
        g = self.gen(lr)
        if hr is None:
            return self.dec(g, lr, lr_feat), None, None
        mu, logvar = self.enc(hr, lr_feat)
        z = mu + torch.exp(0.5 * logvar) * g
        return self.dec(z, lr, lr_feat), mu, logvar


def build_model(a):
    return SRVAE(a.scale, a.lr_size, a.in_ch, a.z_dim, a.nf, getattr(a, "ch_div", 1))     # ch_div is stored in the checkpoint args

### 3.7 Optional VGG feature loss (VGG19 up to conv5_4, avg-pool, first conv changed to 1 input channel)

In [ ]:
class VGGFeat(nn.Module):
    """
    phi_54: VGG19 features up to conv5_4 (frozen), MaxPool -> AvgPool as in the paper.

    First-layer change (3 -> 1 input channel):
        pretrained conv1 weight W has shape (64, 3, 3, 3): one 3x3 filter per colour channel.
        new weight      W_gray = W_R + W_G + W_B   -> shape (64, 1, 3, 3)
        conv(x, W_gray) = conv(x, W_R) + conv(x, W_G) + conv(x, W_B)
    i.e. the new layer reproduces what the pretrained layer would output for an image whose
    R, G, B channels are all equal to x (grayscale), while taking a single-channel input.
    Input normalisation uses the mean of the ImageNet channel means / stds:
        mean = (0.485 + 0.456 + 0.406) / 3 = 0.449,   std = (0.229 + 0.224 + 0.225) / 3 = 0.226
    """

    def __init__(self):
        super().__init__()
        import torchvision
        vgg = torchvision.models.vgg19(weights=torchvision.models.VGG19_Weights.IMAGENET1K_V1).features[:35]

        old = vgg[0]                                                   # Conv2d(3, 64, 3, padding=1)
        new = nn.Conv2d(1, old.out_channels, old.kernel_size, old.stride, old.padding)
        with torch.no_grad():
            new.weight.copy_(old.weight.sum(dim=1, keepdim=True))      # sum over the input-channel axis
            new.bias.copy_(old.bias)
        vgg[0] = new

        for i, m in enumerate(vgg):                                    # MaxPool -> AvgPool (paper)
            if isinstance(m, nn.MaxPool2d):
                vgg[i] = nn.AvgPool2d(2, 2)

        self.net = vgg.eval()
        for p in self.net.parameters():
            p.requires_grad_(False)
        self.register_buffer("mean", torch.tensor(0.449).view(1, 1, 1, 1))
        self.register_buffer("std", torch.tensor(0.226).view(1, 1, 1, 1))

    def forward(self, x):                      # x: (B, 1, H, W) in [0, 1]
        return self.net((x - self.mean) / self.std)

## 4. Losses

L = L1(SR, HR) + lambda * L1(VGG(SR), VGG(HR)) + beta * KL(mu, logvar)

In [ ]:
def kl_standard_normal(mu, logvar):
    """
    Closed-form KL( N(mu, sigma^2 I) || N(0, I) ), per latent dim:
        KL = 0.5 * ( mu^2 + sigma^2 - 1 - log sigma^2 ) = -0.5 * (1 + logvar - mu^2 - exp(logvar))
    This is Eq. 11/13 with mu_P = 0, sigma_P = 1 (the printed equations drop the 1/2 and use
    variances inside the log; the form here is the standard correct one).
    Averaged over batch AND latent dims so kl_weight does not depend on z_dim.
    """
    return -0.5 * torch.mean(1.0 + logvar - mu.pow(2) - logvar.exp())


def total_loss(sr, hr, mu, logvar, vgg, vgg_w, kl_w):
    """
    L = L_SR + L_KL           (Eq. 13)
      = ||Y - Y'||_1 + lambda * ||phi54(Y) - phi54(Y')||_1 + beta * KL      (Eq. 12 + 13)
    """
    l1 = F.l1_loss(sr, hr)
    lv = F.l1_loss(vgg(sr), vgg(hr)) if (vgg is not None and vgg_w > 0) else sr.new_zeros(())
    kl = kl_standard_normal(mu, logvar)
    return l1 + vgg_w * lv + kl_w * kl, l1.detach(), lv.detach(), kl.detach()

## 5. Whole-image metrics (PSNR, SSIM)

No tiles: the model gets the full 128 x 128 image and returns the full 256 x 256 image.

In [ ]:
def check_size(img, lr_size):
    """The encoder / generator FC layers have a fixed input size, so every widefield image must be lr_size x lr_size."""
    if tuple(img.shape[-2:]) != (lr_size, lr_size):
        raise ValueError(f"image is {img.shape[-2]}x{img.shape[-1]} but the model needs exactly {lr_size}x{lr_size} "
                         f"widefield input. Use data of that size, or retrain with a different lr_size.")


def psnr(a, b):
    """PSNR = 10 * log10( MAX^2 / MSE ), MAX = 1 for images in [0,1]."""
    mse = float(np.mean((a - b) ** 2))
    return 10.0 * math.log10(1.0 / max(mse, 1e-12))


def ssim(a, b):
    from skimage.metrics import structural_similarity
    return float(structural_similarity(a, b, data_range=1.0))


@torch.no_grad()
def evaluate(model, pairs, device, max_images=None):
    """Whole-image evaluation: LR (128x128) in, SR (256x256) out, compared with the ground truth. Bicubic is the baseline."""
    model.eval()
    acc = {"psnr": [], "ssim": [], "bic_psnr": [], "bic_ssim": []}
    for lr, hr in pairs[:max_images]:
        check_size(lr, model.lr_size)
        lr_t = torch.from_numpy(lr)[None, None].to(device)
        sr = model(lr_t)[0].clamp(0, 1)[0, 0].cpu().numpy()                      # test path: z = G(X)
        bic = F.interpolate(lr_t, scale_factor=model.scale, mode="bicubic",
                            align_corners=False).clamp(0, 1)[0, 0].cpu().numpy()
        acc["psnr"].append(psnr(sr, hr)); acc["ssim"].append(ssim(sr, hr))
        acc["bic_psnr"].append(psnr(bic, hr)); acc["bic_ssim"].append(ssim(bic, hr))
    return {k: float(np.mean(v)) for k, v in acc.items()}

## 6. Config

Edit `data_root` and `structure`. `lr_size` must equal the size of your widefield images (128).

In [ ]:
cfg = SimpleNamespace(
    data_root=r"C:\Users\BITPATNA\Downloads\vq-VAE_updated",   # folder that contains CCPs / ER / F-actin / Microtubules
    structure="CCPs",                # CCPs | ER | F-actin | Microtubules
    scale=2, in_ch=1,                # BioSR is 2x, grayscale
    lr_size=128,                     # whole widefield image size (SIM ground truth = 2 * 128 = 256)
    z_dim=1024, nf=256,              # paper values for the latent size and the decoder width
    ch_div=8,                        # encoder / LR-stack conv widths = paper widths / ch_div (1 = paper: 32,64,128,256)
    epochs=100, batch=2,             # one epoch = one pass over all training images
    lr=1e-4, weight_decay=0.0,
    vgg_weight=0.0,                  # lambda (0 = off)
    kl_weight=1e-3,                  # beta on the mean-per-dim KL
    val_images=3,                    # validation images scored after every epoch
    workers=0,                       # 0 is safest in notebooks on Windows
    seed=0, out_dir="runs/srvae_full_CCPs",
)
random.seed(cfg.seed); np.random.seed(cfg.seed); torch.manual_seed(cfg.seed)
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", dev)

## 7. Load training and validation pairs (on demand) and check the image sizes

In [ ]:
root = os.path.join(cfg.data_root, cfg.structure)
train_wf, train_gt = os.path.join(root, "training_wf"), os.path.join(root, "training_gt")
val_wf, val_gt = os.path.join(root, "validate_wf"), os.path.join(root, "validate_gt")
for path in (train_wf, train_gt, val_wf, val_gt):
    assert os.path.exists(path), f"Folder not found: {path}"

train_pairs = LazyPairs(train_wf, train_gt)      # instant: only lists the files
val_pairs = LazyPairs(val_wf, val_gt)
print(f"{len(train_pairs)} train / {len(val_pairs)} val pairs (loaded on demand)")

# every image must be exactly lr_size x lr_size (widefield) and scale * lr_size (ground truth)
S = cfg.lr_size
for name, pairs in (("train", train_pairs), ("val", val_pairs)):
    for lr, hr in pairs[:20]:
        assert lr.shape == (S, S) and hr.shape == (S * cfg.scale, S * cfg.scale), \
            f"{name}: widefield {lr.shape} / ground truth {hr.shape}, expected {(S, S)} / {(S * cfg.scale, S * cfg.scale)}"
print(f"sizes OK (checked the first 20 of each): widefield {S}x{S}, ground truth {S * cfg.scale}x{S * cfg.scale}")

lr0, hr0 = train_pairs[0]
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(lr0, cmap="gray"); ax[0].set_title(f"LR widefield {lr0.shape}")
ax[1].imshow(hr0, cmap="gray"); ax[1].set_title(f"HR SIM {hr0.shape}")
for a_ in ax:
    a_.axis("off")
plt.tight_layout(); plt.show()

## 8. Sanity check (size report, then shapes and timing)

The size is first computed on the `meta` device, which allocates no memory. The real model is built only if its weights fit in GPU memory.

In [ ]:
# ---- 1. size report without allocating memory --------------------------------------------------------
with torch.device("meta"):
    _meta = build_model(cfg)
n_par = sum(p.numel() for p in _meta.parameters())
fc1 = _meta.enc.fc[1]
print(f"parameters: {n_par / 1e9:.2f} B | fp32 weights {n_par * 4 / 1e9:.1f} GB | "
      f"weights + gradients + Adam {n_par * 16 / 1e9:.0f} GB | one full checkpoint {n_par * 12 / 1e9:.0f} GB")
print(f"first encoder FC layer: weight {tuple(fc1.weight.shape)} = {fc1.weight.numel() / 1e9:.2f} B parameters")
del _meta
if dev.type == "cuda":
    free, total = torch.cuda.mem_get_info()
    print(f"GPU memory: {free / 1e9:.1f} GB free of {total / 1e9:.1f} GB")
    if n_par * 4 > free:
        raise RuntimeError(f"The weights alone need {n_par * 4 / 1e9:.0f} GB but only {free / 1e9:.1f} GB of GPU memory is free, "
                           f"so the model cannot be built on this GPU.")
    if n_par * 16 > free:
        print(f"WARNING: training needs about {n_par * 16 / 1e9:.0f} GB (weights + gradients + Adam) plus activations, but only "
              f"{free / 1e9:.1f} GB is free, so the training steps below will probably run out of memory.")
else:
    print(f"running on CPU: building the model needs {n_par * 4 / 1e9:.0f} GB of RAM for the weights alone")

# ---- 2. shapes and timing (two real training steps on random data) -----------------------------------
_m = build_model(cfg).to(dev)
S, B = cfg.lr_size, cfg.batch
lr_t = torch.randn(B, cfg.in_ch, S, S, device=dev)
hr_t = torch.randn(B, cfg.in_ch, S * cfg.scale, S * cfg.scale, device=dev)
opt_t = torch.optim.Adam(_m.parameters(), lr=1e-4)
if dev.type == "cuda":
    torch.cuda.reset_peak_memory_stats()
try:
    for step in range(2):                                   # step 0 warms up, step 1 is timed
        if dev.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.time()
        sr, mu, logvar = _m(lr_t, hr_t)                     # training path
        loss, *_ = total_loss(sr, hr_t, mu, logvar, None, 0.0, cfg.kl_weight)
        opt_t.zero_grad(); loss.backward(); opt_t.step()
        if dev.type == "cuda":
            torch.cuda.synchronize()
        dt = time.time() - t0
except torch.cuda.OutOfMemoryError as e:
    raise RuntimeError("Out of memory in a real training step (weights + gradients + Adam + activations do not fit in GPU memory). "
                       "Options: a GPU with more memory, a larger ch_div, or a smaller nf.") from e
assert sr.shape == hr_t.shape and mu.shape == (B, cfg.z_dim) and logvar.shape == (B, cfg.z_dim), (sr.shape, mu.shape)
with torch.no_grad():
    sr_test, _, _ = _m(lr_t)                            # test path
assert sr_test.shape == hr_t.shape
n_it = max(1, len(train_pairs) // B)
print(f"shapes OK: {tuple(lr_t.shape)} + {tuple(hr_t.shape)} -> {tuple(sr.shape)} (training) and -> {tuple(sr_test.shape)} (test)")
print(f"one training step: {dt:.2f} s  ->  about {dt * n_it / 60:.0f} min per epoch ({n_it} iterations; data reading not included)")
if dev.type == "cuda":
    print(f"peak GPU memory: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")
del _m, opt_t, sr, sr_test, mu, logvar, loss
if dev.type == "cuda":
    torch.cuda.empty_cache()

## 9. Train

Resumes automatically from `last.pt` in `out_dir`. Checkpoints are large for this model (`best.pt` about 8.8 GB, `last.pt` about 26 GB), so numbered copies are off by default (`KEEP_EVERY = 0`).

In [ ]:
import shutil

os.makedirs(cfg.out_dir, exist_ok=True)
model = build_model(cfg).to(dev)
print(f"parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f} M")

vgg = VGGFeat().to(dev) if cfg.vgg_weight > 0 else None
opt = torch.optim.Adam(model.parameters(), lr=cfg.lr, betas=(0.9, 0.999), weight_decay=cfg.weight_decay)
sched = torch.optim.lr_scheduler.MultiStepLR(opt, [cfg.epochs // 2], 0.1)   # paper: lr /10 after half
dl = DataLoader(FullImageDataset(train_pairs, augment=True), batch_size=cfg.batch, shuffle=True,
                num_workers=cfg.workers, drop_last=True, pin_memory=dev.type == "cuda")
print(f"batch {cfg.batch} | {len(dl)} iterations per epoch ({len(train_pairs)} whole training images)")

SAVE_EVERY = 2      # full resumable checkpoint (last.pt, overwritten) every N epochs and at the last epoch
KEEP_EVERY = 0      # ALSO keep a numbered copy ckpt_epNNN.pt every N epochs (about 26 GB each). 0 = keep none.
RESUME = True       # True: continue from last.pt if it exists. False: start from scratch.

history, best, start_ep = [], -1.0, 1
last_path = os.path.join(cfg.out_dir, "last.pt")
best_path = os.path.join(cfg.out_dir, "best.pt")


def atomic_save(obj, path):
    """Write to a temp file, then rename, so a crash never leaves a half-written checkpoint."""
    tmp = path + ".tmp"
    torch.save(obj, tmp)
    os.replace(tmp, path)


# ---- resume ------------------------------------------------------------------------------
if RESUME and os.path.isfile(last_path):
    ck = torch.load(last_path, map_location=dev, weights_only=False)
    model.load_state_dict(ck["model"])
    opt.load_state_dict(ck["opt"])
    sched.load_state_dict(ck["sched"])
    history, best, start_ep = ck["history"], ck["best"], ck["epoch"] + 1
    print(f"resumed from epoch {ck['epoch']} (best val PSNR so far {best:.2f})")
    del ck

# ---- training loop -----------------------------------------------------------------------
for ep in range(start_ep, cfg.epochs + 1):
    model.train()
    t0, agg = time.time(), np.zeros(4)
    for lr_b, hr_b in dl:
        lr_b, hr_b = lr_b.to(dev), hr_b.to(dev)
        sr, mu, logvar = model(lr_b, hr_b)
        loss, l1, lv, kl = total_loss(sr, hr_b, mu, logvar, vgg, cfg.vgg_weight, cfg.kl_weight)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        agg += np.array([loss.item(), l1.item(), lv.item(), kl.item()])
    sched.step()
    agg /= len(dl)
    m = evaluate(model, val_pairs, dev, cfg.val_images)
    history.append({"epoch": ep, "loss": float(agg[0]), "l1": float(agg[1]), "kl": float(agg[3]),
                    **{k: float(v) for k, v in m.items()}})
    print(f"ep {ep:03d} | loss {agg[0]:.4f} L1 {agg[1]:.4f} VGG {agg[2]:.4f} KL {agg[3]:.3f} | "
          f"val PSNR {m['psnr']:.2f} (bicubic {m['bic_psnr']:.2f}) SSIM {m['ssim']:.3f} "
          f"(bicubic {m['bic_ssim']:.3f}) | {time.time() - t0:.0f}s")

    # best model: weights only (small, fast to write) - this is what the evaluation cell loads
    if m["psnr"] > best:
        best = m["psnr"]
        atomic_save({"model": model.state_dict(), "args": vars(cfg), "epoch": ep, "metrics": m}, best_path)

    # full resumable state every SAVE_EVERY epochs and at the end
    if ep % SAVE_EVERY == 0 or ep == cfg.epochs:
        atomic_save({"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
                     "args": vars(cfg), "epoch": ep, "metrics": m, "history": history, "best": best},
                    last_path)
        print(f"   checkpoint saved -> {last_path}")
        # numbered copy that is never overwritten
        if KEEP_EVERY and (ep % KEEP_EVERY == 0 or ep == cfg.epochs):
            kept = os.path.join(cfg.out_dir, f"ckpt_ep{ep:03d}.pt")
            shutil.copyfile(last_path, kept)
            print(f"   kept copy        -> {kept}")

## 10. Training curves

In [ ]:
ep = [h["epoch"] for h in history]
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(ep, [h["loss"] for h in history], label="total"); ax[0].plot(ep, [h["l1"] for h in history], label="L1")
ax[0].set_title("train loss"); ax[0].legend()
ax[1].plot(ep, [h["psnr"] for h in history], label="SR-VAE"); ax[1].plot(ep, [h["bic_psnr"] for h in history], "--", label="bicubic")
ax[1].set_title("val PSNR (dB)"); ax[1].legend()
ax[2].plot(ep, [h["ssim"] for h in history], label="SR-VAE"); ax[2].plot(ep, [h["bic_ssim"] for h in history], "--", label="bicubic")
ax[2].set_title("val SSIM"); ax[2].legend()
plt.tight_layout(); plt.show()

## 11. Evaluate the best checkpoint and look at one result

In [ ]:
ck = torch.load(os.path.join(cfg.out_dir, "best.pt"), map_location=dev, weights_only=False)
model = build_model(SimpleNamespace(**ck["args"])).to(dev)
model.load_state_dict(ck["model"]); model.eval()
print(f"loaded best.pt (epoch {ck['epoch']})")
print(evaluate(model, val_pairs, dev))

lr, hr = val_pairs[0]
lr_t = torch.from_numpy(lr)[None, None].to(dev)
with torch.no_grad():
    sr = model(lr_t)[0].clamp(0, 1)[0, 0].cpu().numpy()
bic = F.interpolate(lr_t, scale_factor=model.scale, mode="bicubic", align_corners=False).clamp(0, 1)[0, 0].cpu().numpy()

fig, ax = plt.subplots(1, 4, figsize=(20, 5))
for a_, im, t_ in zip(ax, [lr, bic, sr, hr],
                      ["LR widefield", "Bicubic", f"SR-VAE ({psnr(sr, hr):.2f} dB)", "HR SIM (GT)"]):
    a_.imshow(im, cmap="gray"); a_.set_title(t_); a_.axis("off")
plt.tight_layout(); plt.show()

## 12. Inference on your own widefield image (must be 128 x 128)

In [ ]:
def infer_file(path, out_path="sr.tif"):
    import tifffile
    x = read_stack(path)
    while x.ndim > 2:
        x = x.mean(axis=0)
    check_size(x, model.lr_size)
    x_t = torch.from_numpy(percentile_norm(x))[None, None].to(dev)
    with torch.no_grad():
        out = model(x_t)[0].clamp(0, 1)[0, 0].cpu().numpy()
    tifffile.imwrite(out_path, out)
    return out

# sr_img = infer_file("my_widefield_128.tif", "my_sr.tif")